# 03B Model Validation and Hyperparameter Tuning

> 🟢 **Level A · Required**

A single random-split score is only one exam result. Learn hold-out testing, K-fold CV, overfitting, tuning and leakage.


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline


## Three data roles
Training folds fit model and preprocessing parameters; validation folds choose hyperparameters; the held-out test estimates final performance once. Negative MAE is a scikit-learn scoring convention: larger is better, so negate it for physical error. Fold standard deviation measures variation across these folds, not a confidence interval.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split,KFold,cross_validate,RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
df=pd.read_csv('https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv')
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
X=df[features].copy(); y=df['CO2-1 bar (mol/kg)']
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
cv=KFold(5,shuffle=True,random_state=42)


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
m=RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)
m=make_pipeline(SimpleImputer(strategy='median'), m)
s=cross_validate(m,X_dev,y_dev,cv=cv,scoring={'r2':'r2','mae':'neg_mean_absolute_error'})
print('CV R² =',s['test_r2'].mean(),'+/-',s['test_r2'].std())
print('CV MAE =',(-s['test_mae']).mean(),'+/-',(-s['test_mae']).std())


In [ ]:
param={'n_estimators':[150,300,500],'max_depth':[None,6,10,16],'min_samples_leaf':[1,2,4],'max_features':['sqrt',0.7,1.0]}
param={'randomforestregressor__'+k:v for k,v in param.items()}
search=RandomizedSearchCV(make_pipeline(SimpleImputer(strategy='median'),RandomForestRegressor(random_state=42,n_jobs=1)),param,n_iter=12,cv=cv,scoring='neg_mean_absolute_error',random_state=42,n_jobs=-1).fit(X_dev,y_dev)
p=search.best_estimator_.predict(X_test)
print(search.best_params_)
print('test MAE =',mean_absolute_error(y_test,p),'test R² =',r2_score(y_test,p))


Use CV and tuning only inside the development data. Keep the final test set untouched until model choices are fixed. Common leakage includes near-duplicates, family/topology overlap, preprocessing before splitting and target-derived inputs.

### Completion criterion
Explain `train → CV/tune → untouched test`.


## Family-aware splitting: a concrete contract
Random splitting asks about similar materials from the same population. Group splitting asks about unseen families. A grouping column must come from curated structural/linker/topology metadata; row numbers or arbitrary chunks do not define chemistry. The runnable example demonstrates only the split API; replace its synthetic groups with validated COF families.


In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
toy_X = np.arange(48).reshape(24, 2)
toy_groups = np.repeat(['family_A','family_B','family_C','family_D','family_E','family_F'], 4)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42).split(toy_X, groups=toy_groups))
assert set(toy_groups[tr]).isdisjoint(toy_groups[te])
for train_fold, val_fold in GroupKFold(3).split(toy_X[tr], groups=toy_groups[tr]):
    assert set(toy_groups[tr][train_fold]).isdisjoint(toy_groups[tr][val_fold])
print('train families:', sorted(set(toy_groups[tr])))
print('test families:', sorted(set(toy_groups[te])))


## Exercise
Draw train/CV/test roles before running a search. Explain which population random and group splits evaluate. Record split indices, random seed, preprocessing, and parameter search space. Never choose a random seed because its test score is highest.


## Sources and further reading
[Dataset contracts / 数据使用约定](../../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
